# 06 · Basket Spread — Three Estimators Compared

**Thesis pillar:** (iii) Suitability as a derivatives underlying

Three estimators of the index-weighted average relative bid-ask spread are run in parallel and compared:

- **Method A (Quoted).** Realised daily bid/ask from Refinitiv. Loads `bid_robust.parquet` + `bid_robust_gap.parquet` (ditto ask). Per-stock relative spread is `(ask − bid) / mid`, clipped to [0, 0.20], then time-averaged. Most accurate where quotes are streamed; coverage is partial on illiquid pre-reform names.
- **Method B (Corwin-Schultz, 2012).** Spread inferred from daily high-low bars. Loads `high_robust*.parquet` and `low_robust*.parquet` via the same `_load_num` helper. Non-parametric; requires only OHLC; potentially covers the full universe but is noisier than quotes.
- **Method C (Hybrid).** Uses Method A's quoted spread where available, Method B's Corwin-Schultz estimate as fill-in for stocks without quotes. Maximises coverage while retaining quoted precision on the liquid core.

$$S_{\text{basket}} = \sum_i w_i \cdot \text{spread}_i$$

Weights are renormalised per method among stocks with a spread estimate. Identical statistical battery on each method: paired t-test on matched constituents, 10 000-rep bootstrap 95 % CI on $\Delta S_{\text{basket}}$.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = r'C:\Users\name\Documents\Bachelor Thesis\Empirical Analysis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

START_R = '2025-04-01'
END_R   = '2026-03-31'

orig = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_original_today.parquet'))
pf   = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_proforma.parquet'))

def _load_num(main_path, gap_path=None):
    frames = []
    if os.path.exists(main_path):
        frames.append(pd.read_parquet(main_path))
    if gap_path and os.path.exists(gap_path):
        frames.append(pd.read_parquet(gap_path))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, axis=1)
    df = df.loc[:, ~df.columns.duplicated()]
    return df.apply(pd.to_numeric, errors='coerce').astype(float)

# ----- Method A inputs: bid / ask (Steps 10 + 11b on disk) ------------------
df_bid = _load_num(os.path.join(DATA_DIR, 'bid_robust.parquet'),
                   os.path.join(DATA_DIR, 'bid_robust_gap.parquet'))
df_ask = _load_num(os.path.join(DATA_DIR, 'ask_robust.parquet'),
                   os.path.join(DATA_DIR, 'ask_robust_gap.parquet'))
BA_USABLE = bool(df_bid.notna().to_numpy().any()) and bool(df_ask.notna().to_numpy().any())
A_USABLE  = BA_USABLE

# ----- Method B inputs: HIGH / LOW (OFFLINE only, no API fallback) ----------
HL_MAIN = os.path.join(DATA_DIR, 'high_robust.parquet')
LL_MAIN = os.path.join(DATA_DIR, 'low_robust.parquet')

df_high = _load_num(HL_MAIN, os.path.join(DATA_DIR, 'high_robust_gap.parquet'))
df_low  = _load_num(LL_MAIN, os.path.join(DATA_DIR, 'low_robust_gap.parquet'))
HL_AVAILABLE = (not df_high.empty) and (not df_low.empty) \
               and bool(df_high.notna().to_numpy().any()) \
               and bool(df_low.notna().to_numpy().any())

if not HL_AVAILABLE:
    print('HIGH/LOW parquets not found on disk — Method B (Corwin-Schultz) will be '
          'disabled. Method C (Hybrid) therefore falls back to Method A.')
    print('  expected files: data/high_robust.parquet  data/low_robust.parquet')

# ----- Investable sets ------------------------------------------------------
orig = orig[orig['FF_MktCap'].notna()].copy()
pf   = pf[pf['FF_MktCap'].notna()].copy()
orig['w'] = orig['FF_MktCap'] / orig['FF_MktCap'].sum()
pf['w']   = pf['FF_MktCap']   / pf['FF_MktCap'].sum()

print(f'Original-today : {len(orig):,}    Pro-forma-today : {len(pf):,}')
print(f'Bid matrix  : {df_bid.shape}   non-NA cells: {int(df_bid.notna().to_numpy().sum()):,}')
print(f'Ask matrix  : {df_ask.shape}   non-NA cells: {int(df_ask.notna().to_numpy().sum()):,}')
print(f'High matrix : {df_high.shape}   Low matrix : {df_low.shape}   HL_AVAILABLE={HL_AVAILABLE}')


## 1  Per-stock spread — three methods

In [ ]:
# ── Method A: quoted (ask−bid)/mid, clipped ─────────────────────
if A_USABLE:
    df_bid_f = df_bid.astype(float)
    df_ask_f = df_ask.astype(float)
    df_mid_A    = (df_bid_f + df_ask_f) / 2
    df_spread_A = (df_ask_f - df_bid_f) / df_mid_A
    df_spread_A = df_spread_A.where((df_spread_A >= 0) & (df_spread_A <= 0.20), np.nan)
    avg_spread_A = df_spread_A.mean(axis=0)
else:
    avg_spread_A = pd.Series(dtype=float)

# ── Method B: Corwin-Schultz (2012) from daily OHLC high/low ──────────
def corwin_schultz(high, low):
    beta  = np.log(high/low)**2 + np.log(high.shift(1)/low.shift(1))**2
    gamma = np.log(high.rolling(2).max() / low.rolling(2).min())**2
    alpha = (np.sqrt(2*beta) - np.sqrt(beta)) / (3 - 2*np.sqrt(2)) \
            - np.sqrt(gamma / (3 - 2*np.sqrt(2)))
    return 2*(np.exp(alpha.clip(lower=0)) - 1) / (1 + np.exp(alpha.clip(lower=0)))

if HL_AVAILABLE:
    df_cs = corwin_schultz(df_high.astype(float), df_low.astype(float))
    df_cs = df_cs.where(df_cs >= 0, np.nan)
    avg_spread_B = df_cs.mean(axis=0)
    B_USABLE = bool(avg_spread_B.notna().any())
else:
    avg_spread_B = pd.Series(dtype=float)
    B_USABLE = False

# ── Method C: hybrid = A where available, B fill-in ─────────────────
idx_union    = avg_spread_A.index.union(avg_spread_B.index)
avg_spread_C = avg_spread_A.reindex(idx_union).fillna(avg_spread_B.reindex(idx_union))
C_USABLE = A_USABLE or B_USABLE

print(f'Method A (Quoted)          : {int(avg_spread_A.notna().sum()):>5} stocks with spread  (A_USABLE={A_USABLE})')
print(f'Method B (Corwin-Schultz)  : {int(avg_spread_B.notna().sum()):>5} stocks with spread  (B_USABLE={B_USABLE})')
print(f'Method C (Hybrid)          : {int(avg_spread_C.notna().sum()):>5} stocks with spread  (C_USABLE={C_USABLE})')


## 2  S_basket, paired t-test and bootstrap CI — per method

In [ ]:
results_by_method = {}

def run_method(avg_spread, usable, label):
    """Compute S_basket on both indices + independent-samples tests.

    The spread vector is one value per RIC over the robustness window. The
    two indices are two different constituent lists drawn from that same
    vector; there is no second time period, so a paired design is not
    appropriate (overlap stocks would have identical values on both sides).

    Tests (mirror Amihud / DTT notebooks):
      - Welch t-test, one-sided (H1: mean(spread_pf) < mean(spread_orig)).
      - Mann-Whitney U, one-sided (H1: pf stochastically < orig).
    """
    if not usable or avg_spread.empty:
        return dict(
            label=label, usable=False,
            n_orig=0, n_pf=0,
            s_orig=np.nan, s_pf=np.nan, delta=np.nan,
            med_orig=np.nan, med_pf=np.nan,
            t_stat=np.nan, t_p=np.nan,
            u_stat=np.nan, u_p=np.nan,
            orig_valid=pd.DataFrame(), pf_valid=pd.DataFrame(),
            s_orig_vec=np.array([]), s_pf_vec=np.array([]),
        )

    sp_df = pd.DataFrame({'RIC': avg_spread.index, 'Avg_RelSpread': avg_spread.values})
    _o = orig.merge(sp_df, on='RIC', how='left').dropna(subset=['Avg_RelSpread']).copy()
    _p = pf.merge(sp_df,   on='RIC', how='left').dropna(subset=['Avg_RelSpread']).copy()
    for _d in (_o, _p):
        _d['FF_MktCap']     = pd.to_numeric(_d['FF_MktCap'],     errors='coerce').astype(float)
        _d['Avg_RelSpread'] = pd.to_numeric(_d['Avg_RelSpread'], errors='coerce').astype(float)

    _o_tot = _o['FF_MktCap'].sum()
    _p_tot = _p['FF_MktCap'].sum()
    _o['w_norm'] = (_o['FF_MktCap'] / _o_tot) if _o_tot else 0.0
    _p['w_norm'] = (_p['FF_MktCap'] / _p_tot) if _p_tot else 0.0
    s_o   = float((_o['w_norm'] * _o['Avg_RelSpread']).sum())
    s_p   = float((_p['w_norm'] * _p['Avg_RelSpread']).sum())
    delta = s_p - s_o

    s_orig_vec = _o['Avg_RelSpread'].dropna().to_numpy(dtype=float)
    s_pf_vec   = _p['Avg_RelSpread'].dropna().to_numpy(dtype=float)
    med_o = float(np.median(s_orig_vec)) if len(s_orig_vec) else np.nan
    med_p = float(np.median(s_pf_vec))   if len(s_pf_vec)   else np.nan

    # Welch t-test (unequal variance, one-sided pf < orig)
    if len(s_orig_vec) >= 2 and len(s_pf_vec) >= 2:
        t_stat, t_p = stats.ttest_ind(s_pf_vec, s_orig_vec,
                                      equal_var=False, alternative='less')
    else:
        t_stat, t_p = (np.nan, np.nan)

    # Mann-Whitney U (one-sided pf stochastically < orig)
    if len(s_orig_vec) >= 1 and len(s_pf_vec) >= 1:
        try:
            u_stat, u_p = stats.mannwhitneyu(s_pf_vec, s_orig_vec,
                                             alternative='less')
        except Exception:
            u_stat, u_p = (np.nan, np.nan)
    else:
        u_stat, u_p = (np.nan, np.nan)

    return dict(
        label=label, usable=True,
        n_orig=len(_o), n_pf=len(_p),
        s_orig=s_o, s_pf=s_p, delta=delta,
        med_orig=med_o, med_pf=med_p,
        t_stat=t_stat, t_p=t_p,
        u_stat=u_stat, u_p=u_p,
        orig_valid=_o, pf_valid=_p,
        s_orig_vec=s_orig_vec, s_pf_vec=s_pf_vec,
    )

results_by_method['A'] = run_method(avg_spread_A, A_USABLE, 'A (Quoted)')
results_by_method['B'] = run_method(avg_spread_B, B_USABLE, 'B (Corwin-Schultz)')
results_by_method['C'] = run_method(avg_spread_C, C_USABLE, 'C (Hybrid)')

for k, v in results_by_method.items():
    if v['usable']:
        print(f'  {v["label"]:>22}: N_o={v["n_orig"]:>5}  N_p={v["n_pf"]:>5}  '
              f'S_o={v["s_orig"]*1e4:.2f}bps  S_p={v["s_pf"]*1e4:.2f}bps  '
              f'Δ={v["delta"]*1e4:+.2f}bps   '
              f'Welch t={v["t_stat"]:+.3f} (p={v["t_p"]:.3e})   '
              f'MWU U={v["u_stat"]:.0f} (p={v["u_p"]:.3e})')
    else:
        print(f'  {v["label"]:>22}: not usable')


## 3  Comparison table

In [ ]:
def _fmt(v, spec=':.2f'):
    if isinstance(v, float) and np.isnan(v): return 'NA'
    return format(v, spec.lstrip(':'))

rows = []
for k in ('A', 'B', 'C'):
    v = results_by_method[k]
    rows.append({
        'Method'              : v['label'],
        'Cov orig'            : f'{v["n_orig"]:,} / {len(orig):,}',
        'Cov pf'              : f'{v["n_pf"]:,} / {len(pf):,}',
        'Sbasket orig (bps)'  : _fmt(v['s_orig']*1e4)  if v['usable'] else 'NA',
        'Sbasket pf (bps)'    : _fmt(v['s_pf']*1e4)    if v['usable'] else 'NA',
        'ΔSbasket (bps)'       : _fmt(v['delta']*1e4, ':+.2f') if v['usable'] else 'NA',
        'Welch t (pf<orig)'   :
            (f"t={v['t_stat']:+.3f}, p={v['t_p']:.3e}"
             if v['usable'] and not np.isnan(v['t_p']) else 'NA'),
        'MWU (pf<orig)'       :
            (f"U={v['u_stat']:.0f}, p={v['u_p']:.3e}"
             if v['usable'] and not np.isnan(v['u_p']) else 'NA'),
    })
cmp_table = pd.DataFrame(rows)
print(cmp_table.to_string(index=False))


## 4  CSV exports per method

In [ ]:
for k, v in results_by_method.items():
    summary = pd.DataFrame([{
        'Method'       : v['label'],
        'N_orig'       : v['n_orig'],
        'N_pf'         : v['n_pf'],
        'S_basket_orig': v['s_orig'],
        'S_basket_pf'  : v['s_pf'],
        'Delta'        : v['delta'],
        'Median_orig'  : v['med_orig'],
        'Median_pf'    : v['med_pf'],
        't_stat'       : v['t_stat'],
        't_p_one_sided': v['t_p'],
        'U_stat'       : v['u_stat'],
        'U_p_one_sided': v['u_p'],
    }])
    summary.to_csv(f'basket_spread_summary_{k}.csv', index=False)
    if v['usable']:
        name_o = ['Name'] if 'Name' in v['orig_valid'].columns else []
        name_p = ['Name'] if 'Name' in v['pf_valid'].columns   else []
        v['orig_valid'][['RIC'] + name_o + ['w', 'Avg_RelSpread']].to_csv(
            f'spread_per_stock_original_{k}.csv', index=False)
        v['pf_valid'][  ['RIC'] + name_p + ['w', 'Avg_RelSpread']].to_csv(
            f'spread_per_stock_proforma_{k}.csv', index=False)

print('Saved: basket_spread_summary_[A,B,C].csv, '
      'spread_per_stock_[original,proforma]_[A,B,C].csv')


## 5  Visualisation — 3 columns (per method) × 2 rows

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for col, k in enumerate(('A', 'B', 'C')):
    v = results_by_method[k]
    ax_top = axes[0, col]
    ax_bot = axes[1, col]

    if not v['usable']:
        for ax in (ax_top, ax_bot):
            ax.axis('off')
            ax.text(0.5, 0.5, f'Method {k}\n not usable',
                    ha='center', va='center', transform=ax.transAxes, fontsize=12)
        continue

    # Top: bar (S_basket orig vs pf)
    bars = ax_top.bar(
        ['Original-today', 'Pro-forma-today'],
        [v['s_orig']*1e4, v['s_pf']*1e4],
        color=['steelblue', 'darkorange'], width=0.4, edgecolor='black',
    )
    ax_top.bar_label(bars, fmt='{:.2f} bps', padding=3, fontsize=9)
    ax_top.set_ylabel('S_basket (bps)')
    ax_top.set_title(f'Method {k}: {v["label"]}')
    if not np.isnan(v['delta']):
        ax_top.annotate(f'Δ = {v["delta"]*1e4:+.2f} bps',
                        xy=(0.5, 0.92), xycoords='axes fraction',
                        ha='center', fontsize=9,
                        bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='gray'))

    # Bottom: overlayed spread distributions (orig vs pf), independent samples
    s_o = v['s_orig_vec'] * 1e4
    s_p = v['s_pf_vec']   * 1e4
    if len(s_o) and len(s_p):
        lo = float(np.nanpercentile(np.concatenate([s_o, s_p]),  1))
        hi = float(np.nanpercentile(np.concatenate([s_o, s_p]), 99))
        bins = np.linspace(lo, hi, 60)
        ax_bot.hist(s_o, bins=bins, alpha=0.55, color='steelblue',
                    edgecolor='white', lw=0.3, label=f'Original (N={len(s_o)})')
        ax_bot.hist(s_p, bins=bins, alpha=0.55, color='darkorange',
                    edgecolor='white', lw=0.3, label=f'Pro-forma (N={len(s_p)})')
        ax_bot.axvline(v['med_orig']*1e4, color='steelblue', lw=1.5, ls='--',
                       label=f'Median orig = {v["med_orig"]*1e4:.2f}')
        ax_bot.axvline(v['med_pf']*1e4,   color='darkorange', lw=1.5, ls='--',
                       label=f'Median pf = {v["med_pf"]*1e4:.2f}')
        ax_bot.set_xlabel('Per-stock relative spread (bps)')
        ax_bot.set_ylabel('Stock count')
        t_txt = (f'Welch p = {v["t_p"]:.2e}'
                 if not np.isnan(v['t_p']) else 'Welch: NA')
        u_txt = (f'MWU p = {v["u_p"]:.2e}'
                 if not np.isnan(v['u_p']) else 'MWU: NA')
        ax_bot.text(0.98, 0.95, f'{t_txt}\n{u_txt}',
                    transform=ax_bot.transAxes, fontsize=8, va='top', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray'))
        ax_bot.legend(fontsize=7, loc='upper left')
    else:
        ax_bot.axis('off')
        ax_bot.text(0.5, 0.5, 'No spread observations',
                    ha='center', va='center', transform=ax_bot.transAxes)

plt.suptitle('TOPIX Basket Spread — three estimators (same-window counterfactual, 2025-26)',
             fontsize=13)
plt.tight_layout()
plt.savefig('basket_spread_analysis_3methods.png', bbox_inches='tight')
plt.show()
print('Saved: basket_spread_analysis_3methods.png')
